# 1. LGCA fundamentals and random movement

This lesson builds a complete lattice-gas cellular automaton from
its parts. We will inspect channel states, run unbiased movement,
compare boundary conditions and verify that a seed reproduces the
same stochastic trajectory.

**Learning objectives**

- relate lattice nodes, velocity channels and propagation;
- construct every section of a `ModelSpec`;
- add a random-walk interaction to an `InteractionPipelineSpec`;
- record and plot density and population; and
- use boundaries and seeds as explicit experimental choices.


## States, interaction and propagation

A node stores one Boolean value per channel. An occupied velocity
channel represents a cell that will propagate to the corresponding
neighbor. Rest channels, when present, hold cells at the same node.

A time step first applies the stochastic interaction. Random walk
chooses an admissible channel state without a preferred direction.
Deterministic propagation then moves the velocity-channel cells.
Boundary conditions determine what happens when movement reaches
the edge of the simulated domain.


In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
)
from lgca.pipeline import InteractionPipelineSpec
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder


## A complete first specification

All important choices are visible below. `space` defines the
lattice and boundary, `state` defines initialization, `time`
defines duration and seed, `dynamics` contains the interaction,
and `analysis` states exactly what should be recorded.


In [ ]:
def make_spec(boundary="periodic", seed=11, density=0.15):
    return ModelSpec(
        description=Description(
            title="Seeded random movement",
            details="Unbiased movement on a small square lattice.",
        ),
        space=SpaceSpec(
            geometry="square",
            dims=(18, 18),
            boundary=boundary,
        ),
        state=StateSpec(
            density=density,
            restchannels=0,
        ),
        time=TimeSpec(
            steps=20,
            seed=seed,
        ),
        dynamics=InteractionPipelineSpec(
            operators=[{"name": "classical.random_walk"}],
        ),
        analysis=AnalysisSpec(
            observers=[
                NodeRecorder(),
                DensityRecorder(),
                PopulationRecorder(),
            ],
        ),
    )


spec = make_spec()
spec


Notice that the interaction name is part of the specification.
Nothing is loaded from a pre-built example: replacing or composing
interactions means editing the `dynamics` section you can see.


In [ ]:
one_step = replace(spec, time=replace(spec.time, steps=1))
one_step_result = run_model(one_step, showprogress=False)

before = one_step_result.lgca.nodes_t[0]
after = one_step_result.lgca.nodes_t[1]
print("state shape (x, y, channels):", before.shape)
print("particles before and after:", before.sum(), after.sum())
print("sites whose channel state changed:", np.any(before != after, axis=-1).sum())


Random reorientation and propagation change local channel states,
while this particle-conserving interaction leaves the total number
unchanged. The recorded node arrays exclude boundary ghost nodes,
so their first two axes match the requested 18 by 18 domain.


In [ ]:
result = run_model(spec, showprogress=False)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), constrained_layout=True)
for axis, density_map, title in zip(
    axes,
    (result.lgca.dens_t[0], result.lgca.dens_t[-1]),
    ("initial density", "density after 20 steps"),
):
    image = axis.imshow(density_map.T, origin="lower", vmin=0, vmax=4)
    axis.set_title(title)
    axis.set_xlabel("x")
    axis.set_ylabel("y")
fig.colorbar(image, ax=axes, label="particles per node", shrink=0.8)
plt.show()
plt.close(fig)

print("recorded population:", result.lgca.n_t)


## Boundary conditions are model assumptions

Periodic boundaries connect opposite edges. Reflecting boundaries
represent a no-flux wall. We change only the boundary and seed the
two runs identically, which makes the comparison controlled.


In [ ]:
boundary_results = {
    boundary: run_model(make_spec(boundary=boundary, seed=17), showprogress=False)
    for boundary in ("periodic", "reflecting")
}

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), constrained_layout=True)
for axis, (boundary, boundary_result) in zip(axes, boundary_results.items()):
    axis.imshow(boundary_result.lgca.dens_t[-1].T, origin="lower", vmin=0, vmax=4)
    axis.set_title(boundary)
    axis.set_xlabel("x")
    axis.set_ylabel("y")
plt.show()
plt.close(fig)


With a spatially uniform random initialization, a short run may not
make edge effects dramatic. A useful follow-up experiment is to
initialize cells near one edge and compare escape or accumulation.


## Reproducibility means rerunning the same stochastic history

A seed does not remove stochasticity. It selects one reproducible
realization, which is useful for debugging and paired comparisons.
Scientific uncertainty still requires multiple seeds, introduced
in lesson 5.


In [ ]:
first = run_model(make_spec(seed=42), showprogress=False)
second = run_model(make_spec(seed=42), showprogress=False)
different = run_model(make_spec(seed=43), showprogress=False)

assert np.array_equal(first.lgca.nodes_t, second.lgca.nodes_t)
print("same seed gives identical trajectory:", np.array_equal(first.lgca.nodes_t, second.lgca.nodes_t))
print("different seed gives identical trajectory:", np.array_equal(first.lgca.nodes_t, different.lgca.nodes_t))


## Interpretation

This model represents unbiased motion with excluded channel
occupancy. It is a baseline, not a biological explanation for
directed migration or collective order. Later lessons will add
specific directional mechanisms to the visible pipeline.

## Exercises

1. Change `density` to 0.05 and 0.5. How does crowding affect the
   final density map?
2. Add one rest channel. Does the spatial spread change over the
   same 20 steps?
3. Initialize a compact patch near the left boundary and compare
   periodic, reflecting and absorbing boundaries.
4. Write down the seed and every parameter before sharing a figure.
